<a href="https://colab.research.google.com/github/soumo99/AI_ML_Projects/blob/main/EngieerEfforts_TimeSeriesForecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# PERS Engineer-Specific Effort & Completion Date Forecasting Pipeline
# Target Engineers: Maddock Gunther, Avery Ryan, Savannah Crooks,
#                   Stephanie Keeler, Jason Oberheim, James Hathaway
# Target Horizon: October 2026 through March 2027
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Set plotting style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)

# ------------------------------------------------------------------------------
# 1. LOAD & PREPROCESS DATA
# ------------------------------------------------------------------------------
pers_path = 'PERS Backend 16th Sep.xlsx'
pers_df = pd.read_excel(pers_path)

# Standardize Dates and Numeric Fields
pers_df['Created Date'] = pd.to_datetime(pers_df['Created Date'], errors='coerce')
pers_df['Closed Date'] = pd.to_datetime(pers_df['Closed Date'], errors='coerce')
pers_df['Eng_Total Hours Spent'] = pd.to_numeric(pers_df['Eng_Total Hours Spent'], errors='coerce').fillna(0)

# Target Engineer List
target_engineers = [
    "Maddock, Gunther",
    "Avery, Ryan",
    "Crooks, Savannah",
    "Keeler, Stephanie",
    "Oberheim, Jason",
    "Hathaway, James"
]

# Historical Closed Dataset for Supervised ML Training
closed_mask = pers_df['Status of Request'].isin(['Closed', 'Completed']) & pers_df['Closed Date'].notna() & pers_df['Created Date'].notna()
historical_df = pers_df[closed_mask].copy()
historical_df['Duration_Days'] = (historical_df['Closed Date'] - historical_df['Created Date']).dt.days
historical_df = historical_df[historical_df['Duration_Days'] >= 0]

# Feature Encoding
feature_cols = ['Request Type', 'Business Unit Affected', 'Priority', 'Type of Support Needed', 'Complexity', 'Affected Site']
for col in feature_cols:
    historical_df[col] = historical_df[col].astype(str).fillna('Unknown')
    pers_df[col] = pers_df[col].astype(str).fillna('Unknown')

X_hist = pd.get_dummies(historical_df[feature_cols], drop_first=True)
y_days = historical_df['Duration_Days']
y_hours = historical_df['Eng_Total Hours Spent']

# ------------------------------------------------------------------------------
# 2. TRAIN ML MODELS
# ------------------------------------------------------------------------------
print("Training Machine Learning Regressors...")
model_duration = RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42)
model_duration.fit(X_hist, y_days)

model_hours = GradientBoostingRegressor(n_estimators=150, max_depth=5, learning_rate=0.05, random_state=42)
model_hours.fit(X_hist, y_hours)

# ------------------------------------------------------------------------------
# 3. PREDICT FOR ACTIVE OPEN REQUESTS
# ------------------------------------------------------------------------------
open_mask = ~pers_df['Status of Request'].isin(['Closed', 'Completed', 'Rejected', 'Cancelled'])
open_df = pers_df[open_mask].copy()

X_open = pd.get_dummies(open_df[feature_cols], drop_first=True)
X_open = X_open.reindex(columns=X_hist.columns, fill_value=0)

open_df['Predicted_Duration_Days'] = model_duration.predict(X_open)
open_df['Predicted_Total_Hours'] = np.maximum(0, model_hours.predict(X_open))

today = pd.Timestamp.today().normalize()
open_df['Predicted_Completion_Date'] = open_df.apply(
    lambda r: max(today, r['Created Date']) + pd.to_timedelta(r['Predicted_Duration_Days'], unit='D'),
    axis=1
)

# Filter Open Requests for Target Engineers
eng_open_df = open_df[open_df['Engineer Assigned'].isin(target_engineers)].copy()

# ------------------------------------------------------------------------------
# 4. WEEKLY EFFORT MATRIX GENERATION (TILL MARCH 2027)
# ------------------------------------------------------------------------------
target_horizon_end = pd.Timestamp('2027-03-31')
weekly_timeline = pd.date_range(start=today, end=target_horizon_end, freq='W-MON')

# Create Engineer-level Weekly Matrix
engineer_weekly_matrix = pd.DataFrame(0.0, index=weekly_timeline, columns=target_engineers)

for idx, row in eng_open_df.iterrows():
    eng = row['Engineer Assigned']
    start_dt = max(today, row['Created Date'])
    end_dt = min(target_horizon_end, row['Predicted_Completion_Date'])
    total_hrs = row['Predicted_Total_Hours']

    if end_dt > start_dt and total_hrs > 0:
        active_weeks = pd.date_range(start=start_dt, end=end_dt, freq='W-MON')
        if len(active_weeks) > 0:
            hrs_per_week = total_hrs / len(active_weeks)
            for w in active_weeks:
                if w in engineer_weekly_matrix.index:
                    engineer_weekly_matrix.loc[w, eng] += hrs_per_week

engineer_weekly_matrix = engineer_weekly_matrix.round(2)

# ------------------------------------------------------------------------------
# 5. VISUALIZATION & EXPORT TO EXCEL
# ------------------------------------------------------------------------------
# Plot Weekly Effort per Engineer
engineer_weekly_matrix.plot(kind='line', marker='o', linewidth=2)
plt.title('Forecasted Weekly Engineer Effort (Hours/Week) through March 2027', fontsize=14, fontweight='bold')
plt.xlabel('Week Start Date')
plt.ylabel('Forecasted Hours / Week')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title='Engineer')
plt.tight_layout()
plt.savefig('Engineer_Weekly_Effort_Forecast.png')
plt.show()

# Export Excel Report
output_filename = 'Target_Engineers_Weekly_Forecast_2027.xlsx'
with pd.ExcelWriter(output_filename) as writer:
    eng_open_df[['PERS ID', 'Title of Request (enter details in NOTES section)', 'Status of Request', 'Engineer Assigned', 'Created Date', 'Predicted_Duration_Days', 'Predicted_Completion_Date', 'Predicted_Total_Hours']].to_excel(writer, sheet_name='Engineers_Open_Requests', index=False)
    engineer_weekly_matrix.to_excel(writer, sheet_name='Weekly_Effort_By_Engineer')

print(f"\n✅ Pipeline completed successfully! Detailed forecast saved to: {output_filename}")

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Melt matrix for Plotly compatibility
weekly_melted = engineer_weekly_matrix.reset_index().melt(
    id_vars=['index'],
    var_name='Engineer',
    value_name='Weekly_Hours'
)
weekly_melted.rename(columns={'index': 'Week_Start_Date'}, inplace=True)

# Generate Interactive Plotly Line Chart with Custom Tooltips
fig = px.line(
    weekly_melted,
    x='Week_Start_Date',
    y='Weekly_Hours',
    color='Engineer',
    markers=True,
    title='<b>Weekly Forecasted Engineer Workload Hours (Through March 2027)</b>',
    labels={
        'Week_Start_Date': 'Week Start Date',
        'Weekly_Hours': 'Forecasted Hours / Week',
        'Engineer': 'Engineer Name'
    }
)

# Custom Hover Tooltip Formatting
fig.update_traces(
    hovertemplate="<b>%{fullData.name}</b><br>" +
                  "📅 Week: %{x|%b %d, %Y}<br>" +
                  "⏱️ Weekly Effort: <b>%{y:.2f} hrs</b><extra></extra>"
)

fig.update_layout(
    xaxis_title="<b>Week Start Date</b>",
    yaxis_title="<b>Weekly Effort Allocation (Hours)</b>",
    hovermode="x unified",
    legend_title="<b>Engineers</b>",
    template="plotly_white",
    height=600
)

# Render Chart directly in Google Colab
fig.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor

# 1. Load Data
pers_path = 'PERS Backend 16th Sep.xlsx'
pers_df = pd.read_excel(pers_path)

# Preprocessing
pers_df['Created Date'] = pd.to_datetime(pers_df['Created Date'], errors='coerce')
pers_df['Closed Date'] = pd.to_datetime(pers_df['Closed Date'], errors='coerce')

target_engineers = [
    "Maddock, Gunther",
    "Avery, Ryan",
    "Crooks, Savannah",
    "Keeler, Stephanie",
    "Oberheim, Jason",
    "Hathaway, James"
]

# 2. Train Model on Closed Requests
closed_mask = pers_df['Status of Request'].isin(['Closed', 'Completed']) & pers_df['Closed Date'].notna() & pers_df['Created Date'].notna()
historical_df = pers_df[closed_mask].copy()
historical_df['Duration_Days'] = (historical_df['Closed Date'] - historical_df['Created Date']).dt.days
historical_df = historical_df[historical_df['Duration_Days'] >= 0]

feature_cols = ['Request Type', 'Business Unit Affected', 'Priority', 'Type of Support Needed', 'Complexity', 'Affected Site']
for col in feature_cols:
    historical_df[col] = historical_df[col].astype(str).fillna('Unknown')
    pers_df[col] = pers_df[col].astype(str).fillna('Unknown')

X_hist = pd.get_dummies(historical_df[feature_cols], drop_first=True)
y_days = historical_df['Duration_Days']

model_duration = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42)
model_duration.fit(X_hist, y_days)

# 3. Predict Completion for Open Requests
open_mask = ~pers_df['Status of Request'].isin(['Closed', 'Completed', 'Rejected', 'Cancelled'])
open_df = pers_df[open_mask].copy()

X_open = pd.get_dummies(open_df[feature_cols], drop_first=True)
X_open = X_open.reindex(columns=X_hist.columns, fill_value=0)

open_df['Predicted_Duration_Days'] = model_duration.predict(X_open)

today = pd.Timestamp.today().normalize()
open_df['Predicted_Completion_Date'] = open_df.apply(
    lambda r: max(today, r['Created Date']) + pd.to_timedelta(r['Predicted_Duration_Days'], unit='D'),
    axis=1
)

# 4. Filter for Target Engineers & Summarize Occupation End Month
eng_open_df = open_df[open_df['Engineer Assigned'].isin(target_engineers)].copy()

occupation_summary = []
for eng in target_engineers:
    eng_reqs = eng_open_df[eng_open_df['Engineer Assigned'] == eng]
    if len(eng_reqs) > 0:
        max_date = eng_reqs['Predicted_Completion_Date'].max()
        occupation_summary.append({
            'Engineer': eng,
            'Open Requests': len(eng_reqs),
            'Occupied Until (Exact Date)': max_date.strftime('%B %d, %Y'),
            'Occupied Until (Month)': max_date.strftime('%B %Y')
        })
    else:
        occupation_summary.append({
            'Engineer': eng,
            'Open Requests': 0,
            'Occupied Until (Exact Date)': 'N/A',
            'Occupied Until (Month)': 'Not Currently Occupied'
        })

summary_df = pd.DataFrame(occupation_summary)
print(summary_df.to_string(index=False))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# ------------------------------------------------------------------------------
# 1. LOAD DATA & PREPROCESS
# ------------------------------------------------------------------------------
pers_path = 'PERS Backend 16th Sep.xlsx'
pers_df = pd.read_excel(pers_path)

pers_df['Created Date'] = pd.to_datetime(pers_df['Created Date'], errors='coerce')
pers_df['Closed Date'] = pd.to_datetime(pers_df['Closed Date'], errors='coerce')
pers_df['Eng_Total Hours Spent'] = pd.to_numeric(pers_df['Eng_Total Hours Spent'], errors='coerce').fillna(0)

# Historical Closed Dataset
closed_mask = pers_df['Status of Request'].isin(['Closed', 'Completed']) & pers_df['Closed Date'].notna() & pers_df['Created Date'].notna()
historical_df = pers_df[closed_mask].copy()
historical_df['Duration_Days'] = (historical_df['Closed Date'] - historical_df['Created Date']).dt.days
historical_df = historical_df[historical_df['Duration_Days'] >= 0]

feature_cols = ['Request Type', 'Business Unit Affected', 'Priority', 'Type of Support Needed', 'Complexity', 'Affected Site']
for col in feature_cols:
    historical_df[col] = historical_df[col].astype(str).fillna('Unknown')

X_hist = pd.get_dummies(historical_df[feature_cols], drop_first=True)
y_days = historical_df['Duration_Days']
y_hours = historical_df['Eng_Total Hours Spent']

# ------------------------------------------------------------------------------
# 2. TRAIN & EVALUATE BASELINE MODELS
# ------------------------------------------------------------------------------
X_train, X_test, y_train_days, y_test_days = train_test_split(X_hist, y_days, test_size=0.2, random_state=42)
_, _, y_train_hours, y_test_hours = train_test_split(X_hist, y_hours, test_size=0.2, random_state=42)

# Duration Predictor
model_duration = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42)
model_duration.fit(X_train, y_train_days)
preds_days = model_duration.predict(X_test)

# Effort Hours Predictor
model_hours = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
model_hours.fit(X_train, y_train_hours)
preds_hours = model_hours.predict(X_test)

# Calculate Accuracy Metrics
mae_days = mean_absolute_error(y_test_days, preds_days)
rmse_days = root_mean_squared_error(y_test_days, preds_days)
r2_days = r2_score(y_test_days, preds_days)

mae_hours = mean_absolute_error(y_test_hours, preds_hours)
rmse_hours = root_mean_squared_error(y_test_hours, preds_hours)
r2_hours = r2_score(y_test_hours, preds_hours)

# ------------------------------------------------------------------------------
# 3. BUILD & DISPLAY SCORE SUMMARY TABLE
# ------------------------------------------------------------------------------
score_data = [
    {
        "Model Target": "Completion Duration (Days)",
        "Algorithm": "Random Forest Regressor",
        "MAE": f"{mae_days:.2f} Days",
        "RMSE": f"{rmse_days:.2f} Days",
        "R² Score": f"{r2_days:.4f}",
        "Target Benchmark": "R² ≥ 0.65 | MAE ≤ 25 Days",
        "Status": "Needs Fine-Tuning" if r2_days < 0.60 else "Ready to Proceed"
    },
    {
        "Model Target": "Total Effort (Hours)",
        "Algorithm": "Gradient Boosting Regressor",
        "MAE": f"{mae_hours:.2f} Hours",
        "RMSE": f"{rmse_hours:.2f} Hours",
        "R² Score": f"{r2_hours:.4f}",
        "Target Benchmark": "R² ≥ 0.60 | MAE ≤ 10 Hours",
        "Status": "Needs Fine-Tuning" if r2_hours < 0.60 else "Ready to Proceed"
    }
]

score_df = pd.DataFrame(score_data)

# Display formatted table
print("\n=================== MODEL ACCURACY EVALUATION SCORE TABLE ===================")
print(score_df.to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# ------------------------------------------------------------------------------
# 1. LOAD DATA & DATA CLEANING LOGIC
# ------------------------------------------------------------------------------
pers_path = 'PERS Backend 16th Sep.xlsx'
pers_df = pd.read_excel(pers_path)

pers_df['Created Date'] = pd.to_datetime(pers_df['Created Date'], errors='coerce')
pers_df['Closed Date'] = pd.to_datetime(pers_df['Closed Date'], errors='coerce')
pers_df['Eng_Total Hours Spent'] = pd.to_numeric(pers_df['Eng_Total Hours Spent'], errors='coerce').fillna(0)

# Filter Closed Requests
closed_mask = pers_df['Status of Request'].isin(['Closed', 'Completed']) & pers_df['Closed Date'].notna() & pers_df['Created Date'].notna()
historical_df = pers_df[closed_mask].copy()
historical_df['Duration_Days'] = (historical_df['Closed Date'] - historical_df['Created Date']).dt.days

# Feature Engineering: Add Created Month & Milestone Progress Counter
historical_df['Created_Month'] = historical_df['Created Date'].dt.month.astype(str)
milestone_cols = [c for c in historical_df.columns if c.startswith('M') and 'Date' in c]
historical_df['Milestone_Progress_Count'] = historical_df[milestone_cols].notna().sum(axis=1)

# ------------------------------------------------------------------------------
# 2. FINE-TUNING FILTERING LOGIC
# ------------------------------------------------------------------------------
# A. Duration: Filter negative dates and extreme 95th percentile long-tail outliers
p5_days = historical_df['Duration_Days'].quantile(0.05)
p95_days = historical_df['Duration_Days'].quantile(0.95)
df_dur_tuned = historical_df[(historical_df['Duration_Days'] >= p5_days) & (historical_df['Duration_Days'] <= p95_days)].copy()

# B. Effort Hours: Filter zero-hour unlogged entries and top 2% extreme outliers
p98_hrs = historical_df['Eng_Total Hours Spent'].quantile(0.98)
df_hrs_tuned = historical_df[(historical_df['Eng_Total Hours Spent'] > 0) & (historical_df['Eng_Total Hours Spent'] <= p98_hrs)].copy()

# Feature Encoding
feature_cols = ['Request Type', 'Business Unit Affected', 'Priority', 'Type of Support Needed', 'Complexity', 'Affected Site', 'Engineer Assigned', 'Created_Month', 'Milestone_Progress_Count']

for col in feature_cols:
    df_dur_tuned[col] = df_dur_tuned[col].astype(str).fillna('Unknown')
    df_hrs_tuned[col] = df_hrs_tuned[col].astype(str).fillna('Unknown')

# Log-Transform Targets
X_dur = pd.get_dummies(df_dur_tuned[feature_cols], drop_first=True)
y_dur_log = np.log1p(df_dur_tuned['Duration_Days'])

X_hrs = pd.get_dummies(df_hrs_tuned[feature_cols], drop_first=True)
y_hrs_log = np.log1p(df_hrs_tuned['Eng_Total Hours Spent'])

# Train-Test Split (80/20)
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(X_dur, y_dur_log, test_size=0.2, random_state=42)
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_hrs, y_hrs_log, test_size=0.2, random_state=42)

# ------------------------------------------------------------------------------
# 3. TRAIN FINE-TUNED MODELS
# ------------------------------------------------------------------------------
model_duration = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42)
model_duration.fit(X_train_d, y_train_d)

model_hours = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.03, random_state=42)
model_hours.fit(X_train_h, y_train_h)

# Evaluation Results
preds_dur_log = model_duration.predict(X_test_d)
preds_hrs_log = model_hours.predict(X_test_h)

mae_days = mean_absolute_error(np.expm1(y_test_d), np.expm1(preds_dur_log))
mae_hours = mean_absolute_error(np.expm1(y_test_h), np.expm1(preds_hrs_log))

print("\n=================== FINE-TUNED ACCURACY RESULTS ===================")
print(f"✅ Duration Model MAE: {mae_days:.2f} Days (R² Log Scale: {r2_score(y_test_d, preds_dur_log):.4f})")
print(f"✅ Effort Hours Model MAE: {mae_hours:.2f} Hours (R² Log Scale: {r2_score(y_test_h, preds_hrs_log):.4f})")

Updated Fine Tune logic

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# ------------------------------------------------------------------------------
# 1. LOAD DATA & DATA CLEANING LOGIC
# ------------------------------------------------------------------------------
pers_path = 'PERS Backend 16th Sep.xlsx'
pers_df = pd.read_excel(pers_path)

pers_df['Created Date'] = pd.to_datetime(pers_df['Created Date'], errors='coerce')
pers_df['Closed Date'] = pd.to_datetime(pers_df['Closed Date'], errors='coerce')
pers_df['Eng_Total Hours Spent'] = pd.to_numeric(pers_df['Eng_Total Hours Spent'], errors='coerce').fillna(0)

# Filter Closed Requests for Training
closed_mask = pers_df['Status of Request'].isin(['Closed', 'Completed']) & pers_df['Closed Date'].notna() & pers_df['Created Date'].notna()
historical_df = pers_df[closed_mask].copy()
historical_df['Duration_Days'] = (historical_df['Closed Date'] - historical_df['Created Date']).dt.days

# Feature Engineering: Add Seasonality & Milestone Progress Counter
historical_df['Created_Month'] = historical_df['Created Date'].dt.month.astype(str)
milestone_cols = [c for c in historical_df.columns if c.startswith('M') and 'Date' in c]
historical_df['Milestone_Progress_Count'] = historical_df[milestone_cols].notna().sum(axis=1)

# ------------------------------------------------------------------------------
# 2. FINE-TUNING FILTERING LOGIC
# ------------------------------------------------------------------------------
# A. Duration: Filter negative dates and extreme 95th percentile long-tail outliers
p5_days = historical_df['Duration_Days'].quantile(0.05)
p95_days = historical_df['Duration_Days'].quantile(0.95)
df_dur_tuned = historical_df[(historical_df['Duration_Days'] >= p5_days) & (historical_df['Duration_Days'] <= p95_days)].copy()

# B. Effort Hours: Filter zero-hour unlogged entries and top 2% extreme outliers
p98_hrs = historical_df['Eng_Total Hours Spent'].quantile(0.98)
df_hrs_tuned = historical_df[(historical_df['Eng_Total Hours Spent'] > 0) & (historical_df['Eng_Total Hours Spent'] <= p98_hrs)].copy()

# Categorical & Numerical Feature Set
feature_cols = ['Request Type', 'Business Unit Affected', 'Priority', 'Type of Support Needed', 'Complexity', 'Affected Site', 'Engineer Assigned', 'Created_Month', 'Milestone_Progress_Count']

for col in feature_cols:
    df_dur_tuned[col] = df_dur_tuned[col].astype(str).fillna('Unknown')
    df_hrs_tuned[col] = df_hrs_tuned[col].astype(str).fillna('Unknown')

# Log-Transform Targets
X_dur = pd.get_dummies(df_dur_tuned[feature_cols], drop_first=True)
y_dur_log = np.log1p(df_dur_tuned['Duration_Days'])

X_hrs = pd.get_dummies(df_hrs_tuned[feature_cols], drop_first=True)
y_hrs_log = np.log1p(df_hrs_tuned['Eng_Total Hours Spent'])

# ------------------------------------------------------------------------------
# 3. FIT FINE-TUNED MODELS
# ------------------------------------------------------------------------------
model_dur_tuned = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42)
model_dur_tuned.fit(X_dur, y_dur_log)

model_hrs_tuned = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.03, random_state=42)
model_hrs_tuned.fit(X_hrs, y_hrs_log)

# ------------------------------------------------------------------------------
# 4. PREDICT ON ACTIVE OPEN REQUESTS
# ------------------------------------------------------------------------------
open_mask = ~pers_df['Status of Request'].isin(['Closed', 'Completed', 'Rejected', 'Cancelled'])
open_df = pers_df[open_mask].copy()

open_df['Created_Month'] = open_df['Created Date'].dt.month.astype(str)
open_df['Milestone_Progress_Count'] = open_df[milestone_cols].notna().sum(axis=1)

for col in feature_cols:
    open_df[col] = open_df[col].astype(str).fillna('Unknown')

X_open_dur = pd.get_dummies(open_df[feature_cols], drop_first=True).reindex(columns=X_dur.columns, fill_value=0)
X_open_hrs = pd.get_dummies(open_df[feature_cols], drop_first=True).reindex(columns=X_hrs.columns, fill_value=0)

# Reverse Log Transformation using expm1
open_df['Pred_Duration_Days'] = np.expm1(model_dur_tuned.predict(X_open_dur))
open_df['Pred_Total_Hours'] = np.expm1(model_hrs_tuned.predict(X_open_hrs))

today = pd.Timestamp.today().normalize()
open_df['Pred_Completion_Date'] = open_df.apply(
    lambda r: max(today, r['Created Date']) + pd.to_timedelta(r['Pred_Duration_Days'], unit='D'),
    axis=1
)

# ------------------------------------------------------------------------------
# 5. GENERATE WEEKLY EFFORT MATRIX & PLOT LINE CHART
# ------------------------------------------------------------------------------
target_engineers = [
    "Maddock, Gunther",
    "Avery, Ryan",
    "Crooks, Savannah",
    "Keeler, Stephanie",
    "Oberheim, Jason",
    "Hathaway, James"
]

eng_open_df = open_df[open_df['Engineer Assigned'].isin(target_engineers)].copy()

target_horizon_end = pd.Timestamp('2027-03-31')
weekly_timeline = pd.date_range(start=today, end=target_horizon_end, freq='W-MON')

finetuned_weekly_matrix = pd.DataFrame(0.0, index=weekly_timeline, columns=target_engineers)

for idx, row in eng_open_df.iterrows():
    eng = row['Engineer Assigned']
    start_dt = max(today, row['Created Date'])
    end_dt = min(target_horizon_end, row['Pred_Completion_Date'])
    total_hrs = row['Pred_Total_Hours']

    if end_dt > start_dt and total_hrs > 0:
        active_weeks = pd.date_range(start=start_dt, end=end_dt, freq='W-MON')
        if len(active_weeks) > 0:
            hrs_per_week = total_hrs / len(active_weeks)
            for w in active_weeks:
                if w in finetuned_weekly_matrix.index:
                    finetuned_weekly_matrix.loc[w, eng] += hrs_per_week

finetuned_weekly_matrix = finetuned_weekly_matrix.round(2)

# Plot Line Chart View
plt.figure(figsize=(15, 8), dpi=120)

for column in finetuned_weekly_matrix.columns:
    if finetuned_weekly_matrix[column].sum() > 0:
        plt.plot(
            finetuned_weekly_matrix.index.strftime('%b %d, %Y'),
            finetuned_weekly_matrix[column],
            marker='o',
            linewidth=2.5,
            label=f"{column} (Total: {finetuned_weekly_matrix[column].sum():.1f} hrs)"
        )

plt.title('Fine-Tuned Forecast: Weekly Engineer Workload Hours (Through March 2027)', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Week Start Date', fontsize=12, labelpad=10)
plt.ylabel('Weekly Allocation (Hours)', fontsize=12, labelpad=10)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(title='Engineer Workload', fontsize=11, title_fontsize='12', loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# 1. Melt matrix for Plotly compatibility
weekly_melted = finetuned_weekly_matrix.reset_index().melt(
    id_vars=['index'],
    var_name='Engineer',
    value_name='Weekly_Hours'
)
weekly_melted.rename(columns={'index': 'Week_Start_Date'}, inplace=True)

# 2. Build Interactive Plotly Line Chart
fig = px.line(
    weekly_melted,
    x='Week_Start_Date',
    y='Weekly_Hours',
    color='Engineer',
    markers=True,
    title='<b>Fine-Tuned Forecast: Weekly Engineer Workload Hours (Through March 2027)</b>',
    labels={
        'Week_Start_Date': 'Week Start Date',
        'Weekly_Hours': 'Forecasted Hours / Week',
        'Engineer': 'Engineer Name'
    }
)

# 3. Custom Hover Tooltip Formatting
fig.update_traces(
    hovertemplate="<b>%{fullData.name}</b><br>" +
                  "📅 Week: %{x|%b %d, %Y}<br>" +
                  "⏱️ Weekly Effort: <b>%{y:.2f} hrs</b><extra></extra>"
)

# 4. Interactive Layout Configuration
fig.update_layout(
    xaxis_title="<b>Week Start Date</b>",
    yaxis_title="<b>Weekly Allocation (Hours)</b>",
    hovermode="x unified",  # Unified hover shows all active engineers simultaneously for that week
    legend_title="<b>Engineers</b>",
    template="plotly_white",
    height=600
)

# 5. Display Interactive Chart directly in Colab
fig.show()

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# ------------------------------------------------------------------------------
# 1. LOAD DATASETS
# ------------------------------------------------------------------------------
pers_path = 'PERS Backend 16th Sep.xlsx'
tracker_path = 'Time Tracker Backend 16thSep.xlsx'

pers_df = pd.read_excel(pers_path)
tracker_df = pd.read_excel(tracker_path)

target_engineers = [
    "Maddock, Gunther",
    "Avery, Ryan",
    "Crooks, Savannah",
    "Keeler, Stephanie",
    "Oberheim, Jason"
]

# ------------------------------------------------------------------------------
# 2. EXTRACT ACTUAL LOGGED WEEKLY HOURS (TIME TRACKER)
# ------------------------------------------------------------------------------
tracker_eng = tracker_df[tracker_df['Engineer'].isin(target_engineers)].copy()

# Extract weekly columns (Week_1 to Week_39)
week_cols = [f'Week_{i}' for i in range(1, 40) if f'Week_{i}' in tracker_df.columns]
actual_weekly_matrix = tracker_eng.groupby('Engineer')[week_cols].sum().T

# Map Week 1..39 to 2026 Calendar Week Start Dates
actual_week_dates = pd.date_range(start='2026-01-05', periods=len(week_cols), freq='W-MON')
actual_weekly_matrix.index = actual_week_dates

# ------------------------------------------------------------------------------
# 3. FINE-TUNED ML PREDICTIONS (FORECAST FOR OPEN REQUESTS)
# ------------------------------------------------------------------------------
pers_df['Created Date'] = pd.to_datetime(pers_df['Created Date'], errors='coerce')
pers_df['Closed Date'] = pd.to_datetime(pers_df['Closed Date'], errors='coerce')
pers_df['Eng_Total Hours Spent'] = pd.to_numeric(pers_df['Eng_Total Hours Spent'], errors='coerce').fillna(0)

# Historical Closed Training Set
closed_mask = pers_df['Status of Request'].isin(['Closed', 'Completed']) & pers_df['Closed Date'].notna() & pers_df['Created Date'].notna()
historical_df = pers_df[closed_mask].copy()
historical_df['Duration_Days'] = (historical_df['Closed Date'] - historical_df['Created Date']).dt.days

historical_df['Created_Month'] = historical_df['Created Date'].dt.month.astype(str)
milestone_cols = [c for c in historical_df.columns if c.startswith('M') and 'Date' in c]
historical_df['Milestone_Progress_Count'] = historical_df[milestone_cols].notna().sum(axis=1)

# Outlier Trimming & Filters
p5_days, p95_days = historical_df['Duration_Days'].quantile(0.05), historical_df['Duration_Days'].quantile(0.95)
df_dur_tuned = historical_df[(historical_df['Duration_Days'] >= p5_days) & (historical_df['Duration_Days'] <= p95_days)].copy()

p98_hrs = historical_df['Eng_Total Hours Spent'].quantile(0.98)
df_hrs_tuned = historical_df[(historical_df['Eng_Total Hours Spent'] > 0) & (historical_df['Eng_Total Hours Spent'] <= p98_hrs)].copy()

feature_cols = ['Request Type', 'Business Unit Affected', 'Priority', 'Type of Support Needed', 'Complexity', 'Affected Site', 'Engineer Assigned', 'Created_Month', 'Milestone_Progress_Count']

for col in feature_cols:
    df_dur_tuned[col] = df_dur_tuned[col].astype(str).fillna('Unknown')
    df_hrs_tuned[col] = df_hrs_tuned[col].astype(str).fillna('Unknown')

X_dur = pd.get_dummies(df_dur_tuned[feature_cols], drop_first=True)
y_dur_log = np.log1p(df_dur_tuned['Duration_Days'])

X_hrs = pd.get_dummies(df_hrs_tuned[feature_cols], drop_first=True)
y_hrs_log = np.log1p(df_hrs_tuned['Eng_Total Hours Spent'])

# Train Fine-Tuned Regressors
model_duration = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42)
model_duration.fit(X_dur, y_dur_log)

model_hours = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.03, random_state=42)
model_hours.fit(X_hrs, y_hrs_log)

# Open Requests Workload Forecast (Oct 2026 - March 2027)
open_mask = ~pers_df['Status of Request'].isin(['Closed', 'Completed', 'Rejected', 'Cancelled'])
open_df = pers_df[open_mask].copy()

open_df['Created_Month'] = open_df['Created Date'].dt.month.astype(str)
open_df['Milestone_Progress_Count'] = open_df[milestone_cols].notna().sum(axis=1)

for col in feature_cols:
    open_df[col] = open_df[col].astype(str).fillna('Unknown')

X_open_dur = pd.get_dummies(open_df[feature_cols], drop_first=True).reindex(columns=X_dur.columns, fill_value=0)
X_open_hrs = pd.get_dummies(open_df[feature_cols], drop_first=True).reindex(columns=X_hrs.columns, fill_value=0)

open_df['Pred_Duration_Days'] = np.expm1(model_duration.predict(X_open_dur))
open_df['Pred_Total_Hours'] = np.expm1(model_hours.predict(X_open_hrs))

today = pd.Timestamp.today().normalize()
open_df['Pred_Completion_Date'] = open_df.apply(
    lambda r: max(today, r['Created Date']) + pd.to_timedelta(r['Pred_Duration_Days'], unit='D'),
    axis=1
)

eng_open_df = open_df[open_df['Engineer Assigned'].isin(target_engineers)].copy()

target_horizon_end = pd.Timestamp('2027-03-31')
forecast_week_dates = pd.date_range(start=today, end=target_horizon_end, freq='W-MON')

predicted_weekly_matrix = pd.DataFrame(0.0, index=forecast_week_dates, columns=target_engineers)

for idx, row in eng_open_df.iterrows():
    eng = row['Engineer Assigned']
    start_dt = max(today, row['Created Date'])
    end_dt = min(target_horizon_end, row['Pred_Completion_Date'])
    total_hrs = row['Pred_Total_Hours']

    if end_dt > start_dt and total_hrs > 0:
        active_weeks = pd.date_range(start=start_dt, end=end_dt, freq='W-MON')
        if len(active_weeks) > 0:
            hrs_per_week = total_hrs / len(active_weeks)
            for w in active_weeks:
                if w in predicted_weekly_matrix.index:
                    predicted_weekly_matrix.loc[w, eng] += hrs_per_week

predicted_weekly_matrix = predicted_weekly_matrix.round(2)

# ------------------------------------------------------------------------------
# 4. BUILD INTERACTIVE ACTUAL VS. FORECASTED LINE CHART
# ------------------------------------------------------------------------------
fig = go.Figure()

engineer_colors = {
    "Maddock, Gunther": "#1f77b4",
    "Avery, Ryan": "#ff7f0e",
    "Crooks, Savannah": "#2ca02c",
    "Keeler, Stephanie": "#d62728",
    "Oberheim, Jason": "#9467bd"
}

for eng in target_engineers:
    # Actual Historical Logs (Solid Line)
    if eng in actual_weekly_matrix.columns:
        fig.add_trace(go.Scatter(
            x=actual_weekly_matrix.index,
            y=actual_weekly_matrix[eng],
            mode='lines+markers',
            name=f'{eng} (Actual Logged)',
            line=dict(color=engineer_colors[eng], width=2.5),
            marker=dict(symbol='circle', size=6),
            hovertemplate="<b>%{fullData.name}</b><br>📅 Week: %{x|%b %d, %Y}<br>⏱️ Actual Logged: <b>%{y:.1f} hrs</b><extra></extra>"
        ))

    # Forecasted Effort (Dashed Line)
    if eng in predicted_weekly_matrix.columns:
        fig.add_trace(go.Scatter(
            x=predicted_weekly_matrix.index,
            y=predicted_weekly_matrix[eng],
            mode='lines+markers',
            name=f'{eng} (Forecasted Effort)',
            line=dict(color=engineer_colors[eng], width=2.5, dash='dash'),
            marker=dict(symbol='square', size=6),
            hovertemplate="<b>%{fullData.name}</b><br>📅 Week: %{x|%b %d, %Y}<br>⏱️ Forecasted: <b>%{y:.2f} hrs</b><extra></extra>"
        ))

fig.update_layout(
    title="<b>Actual Logged vs. Forecasted Weekly Engineer Workload (2026 – March 2027)</b>",
    xaxis_title="<b>Week Start Date</b>",
    yaxis_title="<b>Weekly Effort Allocation (Hours)</b>",
    template="plotly_white",
    hovermode="x unified",
    legend_title="<b>Engineer Allocation</b>",
    height=650
)

# Render interactive Plotly chart directly in Google Colab
fig.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from datetime import datetime, timedelta

# ==============================================================================
# 1. LOAD DATASETS & FILTER SPECIFIED ENGINEERS
# ==============================================================================
tt_file = 'Time Tracker Backend 5thOct.xlsx'
pers_file = 'PERS Backend 5thOct.xlsx'

df_tt = pd.read_excel(tt_file, sheet_name=0)
df_pers = pd.read_excel(pers_file, sheet_name=0)

requested_engineers = [
    'Avery, Ryan',
    'Maddock, Gunther',
    'Keeler, Stephanie',
    'Boyer, Rick',
    'Crooks, Savannah',
    'Rivera, Tina'
]

weeks_16_40 = [f'Week_{w}' for w in range(16, 41)]
forecast_weeks = list(range(41, 53))
all_weeks = [f"Week {w}" for w in range(16, 53)]
x_indices = np.arange(len(all_weeks))

df_req = df_tt[df_tt['Engineer'].isin(requested_engineers)].copy()

# ==============================================================================
# 2. INDIVIDUAL RANDOM FOREST MODELING & PLOTTING
# ==============================================================================
fig, axes = plt.subplots(3, 2, figsize=(16, 12), dpi=300, sharex=True)
axes = axes.flatten()

engineer_forecast_results = {}

for idx, eng in enumerate(requested_engineers):
    eng_df = df_req[df_req['Engineer'] == eng]

    # Extract actual weekly hours (Weeks 16 to 40)
    if len(eng_df) > 0:
        actual_vals = eng_df[weeks_16_40].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=0).values
    else:
        actual_vals = np.zeros(len(weeks_16_40))

    # Build time-series lag feature matrix
    X_train, y_train = [], []
    for i in range(3, len(actual_vals)):
        lag1 = actual_vals[i-1]
        lag2 = actual_vals[i-2]
        lag3 = actual_vals[i-3]
        roll_mean = np.mean([lag1, lag2, lag3])
        X_train.append([i + 16, lag1, lag2, lag3, roll_mean])
        y_train.append(actual_vals[i])

    X_df = pd.DataFrame(X_train, columns=['week_num', 'lag1', 'lag2', 'lag3', 'roll_mean'])
    y_arr = np.array(y_train)

    # Train model
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X_df, y_arr)

    history = list(actual_vals)
    baseline_velocity = np.mean(history[-8:-1]) if np.mean(history[-8:-1]) > 0 else 35.0

    # Recursive autoregressive forecasting for Weeks 41 to 52
    forecast_vals = []
    for w in forecast_weeks:
        lag1 = history[-1]
        lag2 = history[-2]
        lag3 = history[-3]
        roll_mean = np.mean([lag1, lag2, lag3])
        feat = pd.DataFrame([[w, lag1, lag2, lag3, roll_mean]], columns=['week_num', 'lag1', 'lag2', 'lag3', 'roll_mean'])
        pred = rf.predict(feat)[0]
        smoothed_pred = 0.6 * pred + 0.4 * baseline_velocity
        forecast_vals.append(smoothed_pred)
        history.append(smoothed_pred)

    engineer_forecast_results[eng] = forecast_vals

    # Plot individual subplot
    ax = axes[idx]
    ax.plot(x_indices[:25], actual_vals, marker='o', linewidth=2, color='#1f77b4', label='Actual Hours')
    ax.plot(x_indices[24:], [actual_vals[-1]] + forecast_vals, marker='s', linestyle='--', linewidth=2, color='#ff7f0e', label='Forecasted Hours')

    ax.axvline(x=24, color='gray', linestyle=':', linewidth=1.2)
    ax.set_title(eng, fontsize=12, fontweight='bold')
    ax.set_ylabel('Hours (hrs)', fontsize=10)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(fontsize=9, loc='upper left')

for ax in axes[-2:]:
    ax.set_xticks(x_indices[::2])
    ax.set_xticklabels(all_weeks[::2], rotation=45, fontsize=9)
    ax.set_xlabel('Week Number', fontsize=10)

plt.suptitle('Requested Engineers Weekly Effort: Actual vs. Forecasted (Weeks 16–52)', fontsize=15, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('requested_engineers_weekly_effort.png')
plt.show()